# 02 — Обучение моделей

Пайплайн:
1. Загрузка / генерация данных
2. Feature Engineering
3. Train / Test split
4. LightGBM + Feature Importance
5. Autoencoder + Reconstruction Error
6. Stacking Ensemble
7. Сравнение моделей
8. SHAP анализ топ-10 фичей

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix, f1_score,
)

from kyt_engine.data.elliptic import load_elliptic
from kyt_engine.features.engine import FeatureEngineer
from kyt_engine.models import LightGBMClassifier, AutoencoderDetector, StackingEnsemble

RANDOM_STATE = 42
DATA_DIR = Path("../data/raw")
print("Imports OK")

## 1. Загрузка и подготовка данных

In [ ]:
raw = load_elliptic(DATA_DIR)

nodes = raw["nodes"]
edges = raw["edges"]
classes = raw["classes"]

print(f"Nodes:  {nodes.shape}")
print(f"Edges:  {edges.shape}")
print(f"Classes: {classes.shape}")
print(f"Label distribution:\n{classes["label"].value_counts().sort_index()}")
nodes.head()

In [ ]:
df = nodes.merge(classes, on="txId", how="left")
df.head()

In [ ]:
fe = FeatureEngineer()
features = fe.fit_transform(df)
labels = df.loc[features.index, "label"].fillna(0).astype(int)

print(f"Features: {features.shape}")
print(f"Label balance: {labels.value_counts().to_dict()}")
features.head()

## 2. Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=labels,
)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Train fraud ratio: {y_train.mean():.4f}")
print(f"Test fraud ratio:  {y_test.mean():.4f}")

## 3. LightGBM

In [ ]:
lgbm = LightGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    random_state=RANDOM_STATE,
)
lgbm.fit(X_train, y_train)

lgbm_proba = lgbm.predict_proba(X_test)[:, 1]
lgbm_preds = lgbm.predict(X_test)

print(f"Threshold: {lgbm.threshold:.3f}")
print(classification_report(y_test, lgbm_preds, target_names=["licit", "illicit"]))

### Feature Importance (LightGBM)

In [ ]:
importances = lgbm._model.feature_importances_
feat_names = np.array(lgbm.feature_names)
idx = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(idx)), importances[idx][::-1], color=sns.color_palette("Set2", len(idx)))
ax.set_yticks(range(len(idx)))
ax.set_yticklabels(feat_names[idx][::-1], fontsize=10)
ax.set_xlabel("Importance (split count)", fontsize=12)
ax.set_title("LightGBM — Top-20 Feature Importance", fontsize=14, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
gain_imp = lgbm._model.booster_.feature_importance(importance_type="gain")
idx_gain = np.argsort(gain_imp)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(idx_gain)), gain_imp[idx_gain][::-1], color=sns.color_palette("flare", len(idx_gain)))
ax.set_yticks(range(len(idx_gain)))
ax.set_yticklabels(feat_names[idx_gain][::-1], fontsize=10)
ax.set_xlabel("Gain", fontsize=12)
ax.set_title("LightGBM — Top-20 Feature Importance (Gain)", fontsize=14, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Autoencoder

In [ ]:
ae = AutoencoderDetector(
    latent_dim=32,
    epochs=50,
    batch_size=64,
    contamination=0.05,
    random_state=RANDOM_STATE,
)
ae.fit(X_train, y_train)

ae_proba = ae.predict_proba(X_test)[:, 1]
ae_preds = ae.predict(X_test)

print(f"Threshold: {ae.threshold:.6f}")
print(classification_report(y_test, ae_preds, target_names=["licit", "illicit"]))

### Reconstruction Error

In [ ]:
X_test_clean = X_test.replace([np.inf, -np.inf], np.nan).fillna(0.0)
X_test_np = X_test_clean.to_numpy(dtype=np.float32)
X_test_scaled = (X_test_np - ae._mean) / ae._std

import torch

ae._net.eval()
with torch.no_grad():
    x_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(ae._device)
    recon = ae._net(x_t)
    recon_errors = torch.mean((recon.cpu() - x_t.cpu()) ** 2, dim=1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, "#2ecc71", "licit"), (1, "#e74c3c", "illicit")]:
    mask = y_test.values == label
    axes[0].hist(recon_errors[mask], bins=60, alpha=0.65, color=color, label=name, density=True)
axes[0].axvline(ae.threshold, color="black", linestyle="--", linewidth=1.5, label=f"threshold={ae.threshold:.4f}")
axes[0].set_xlabel("Reconstruction Error (MSE)", fontsize=11)
axes[0].set_ylabel("Density", fontsize=11)
axes[0].set_title("Autoencoder — Reconstruction Error Distribution", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, np.percentile(recon_errors, 99))

for label, color, name in [(0, "#2ecc71", "licit"), (1, "#e74c3c", "illicit")]:
    mask = y_test.values == label
    sorted_err = np.sort(recon_errors[mask])
    cdf = np.arange(1, len(sorted_err) + 1) / len(sorted_err)
    axes[1].plot(sorted_err, cdf, color=color, label=name, linewidth=1.5)
axes[1].axvline(ae.threshold, color="black", linestyle="--", linewidth=1.5, label=f"threshold")
axes[1].set_xlabel("Reconstruction Error (MSE)", fontsize=11)
axes[1].set_ylabel("CDF", fontsize=11)
axes[1].set_title("Autoencoder — CDF by Class", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].set_xlim(0, np.percentile(recon_errors, 99))

plt.tight_layout()
plt.show()

## 5. Stacking Ensemble

In [ ]:
ensemble = StackingEnsemble(
    lgbm_params={"n_estimators": 500, "learning_rate": 0.05, "num_leaves": 63},
    ae_params={"latent_dim": 32, "epochs": 50, "batch_size": 64, "contamination": 0.05},
    random_state=RANDOM_STATE,
)
ensemble.fit(X_train, y_train)

ens_proba = ensemble.predict_proba(X_test)[:, 1]
ens_preds = ensemble.predict(X_test)

print(f"Threshold: {ensemble.threshold:.3f}")
print(classification_report(y_test, ens_preds, target_names=["licit", "illicit"]))

## 6. Сравнение моделей

In [ ]:
results = {
    "LightGBM":    {"proba": lgbm_proba, "preds": lgbm_preds},
    "Autoencoder": {"proba": ae_proba,    "preds": ae_preds},
    "Ensemble":    {"proba": ens_proba,   "preds": ens_preds},
}

metrics_rows = []
for name, r in results.items():
    auc = roc_auc_score(y_test, r["proba"])
    ap = average_precision_score(y_test, r["proba"])
    f1 = f1_score(y_test, r["preds"])
    metrics_rows.append({"Model": name, "ROC-AUC": auc, "PR-AUC (AP)": ap, "F1": f1})

metrics_df = pd.DataFrame(metrics_rows).set_index("Model")
metrics_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

palette = {"LightGBM": "#3498db", "Autoencoder": "#e67e22", "Ensemble": "#2ecc71"}

# --- ROC ---
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["proba"])
    auc_val = roc_auc_score(y_test, r["proba"])
    axes[0].plot(fpr, tpr, color=palette[name], linewidth=2, label=f"{name} (AUC={auc_val:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3, linewidth=1)
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate", fontsize=12)
axes[0].set_title("ROC-кривая", fontsize=14, fontweight="bold")
axes[0].legend(fontsize=11, loc="lower right")
axes[0].set_xlim([-0.01, 1.01])
axes[0].set_ylim([-0.01, 1.01])
axes[0].grid(True, alpha=0.3)

# --- PR ---
for name, r in results.items():
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, r["proba"])
    ap_val = average_precision_score(y_test, r["proba"])
    axes[1].plot(rec_arr, prec_arr, color=palette[name], linewidth=2, label=f"{name} (AP={ap_val:.4f})")
baseline = y_test.mean()
axes[1].axhline(baseline, color="black", linestyle="--", alpha=0.3, linewidth=1, label=f"Baseline={baseline:.3f}")
axes[1].set_xlabel("Recall", fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].set_title("PR-кривая", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=11, loc="upper right")
axes[1].set_xlim([-0.01, 1.01])
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r["preds"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["licit", "illicit"],
                yticklabels=["licit", "illicit"],
                cbar=False)
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrix — все модели", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r["preds"], normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2%", cmap="Oranges", ax=ax,
                xticklabels=["licit", "illicit"],
                yticklabels=["licit", "illicit"],
                cbar=False)
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrix (normalized) — все модели", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 7. SHAP анализ — топ-10 фичей

In [ ]:
import shap

explainer = shap.TreeExplainer(lgbm._model)
shap_values = explainer.shap_values(X_test)

# Для бинарной классификации shap_values — массив [class_0, class_1]
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

print(f"SHAP values shape: {np.array(shap_vals).shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_vals, X_test,
    max_display=10,
    show=False,
    plot_size=None,
)
plt.title("SHAP Summary — Top-10 Features (LightGBM)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
mean_abs = np.abs(shap_vals).mean(axis=0)
top10_idx = np.argsort(mean_abs)[::-1][:10]
top10_names = np.array(X_test.columns)[top10_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(10), mean_abs[top10_idx][::-1], color=sns.color_palette("viridis", 10))
ax.set_yticks(range(10))
ax.set_yticklabels(top10_names[::-1], fontsize=11)
ax.set_xlabel("mean(|SHAP value|)", fontsize=12)
ax.set_title("SHAP — Top-10 Most Important Features", fontsize=14, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
top2 = top10_names[:2]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, feat in zip(axes, top2):
    vals = X_test[feat].values
    sv = shap_vals[:, list(X_test.columns).index(feat)]
    sc = ax.scatter(vals, sv, c=y_test.values, cmap="RdYlGn_r", alpha=0.5, s=10)
    ax.set_xlabel(feat, fontsize=11)
    ax.set_ylabel("SHAP value", fontsize=11)
    ax.set_title(f"SHAP Dependence — {feat}", fontsize=12, fontweight="bold")
    ax.axhline(0, color="gray", linestyle="--", alpha=0.4)
    plt.colorbar(sc, ax=ax, label="label (1=illicit)")

plt.tight_layout()
plt.show()

## 8. Краткие выводы

In [ ]:
print("=" * 60)
print("СВОДКА МЕТРИК")
print("=" * 60)
print(metrics_df.round(4).to_string())
print()

best_model = metrics_df["F1"].idxmax()
print(f"Лучшая модель по F1: {best_model} ({metrics_df.loc[best_model, 'F1']:.4f})")
print()
print("Ключевые SHAP-фичи:", ", ".join(top10_names[:5].tolist()))